# **Celda 1: Carga, limpieza y preparación de datos**

In [ ]:
# Celda 1: Carga, limpieza y preparación de datos (CORREGIDA)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
df = pd.read_csv('dataset_salud_nayarit_limpio.csv')

# Limpieza de la variable objetivo
def clean_hospitalizacion(valor):
    if pd.isna(valor):
        return 0
    valor_str = str(valor).upper().strip()
    if valor_str in ['1', 'INGRESADO']:
        return 1
    else:
        return 0

df['hospitalizacion'] = df['hospitalizacion'].apply(clean_hospitalizacion)

print("="*60)
print("DATASET CARGADO Y LIMPIO")
print("="*60)
print(f"Total de registros: {len(df)}")
print(f"Hospitalizados (1): {df['hospitalizacion'].sum()}")
print(f"No hospitalizados (0): {len(df) - df['hospitalizacion'].sum()}")

# Separar características y objetivo
X = df.drop('hospitalizacion', axis=1)
y = df['hospitalizacion']

# Identificar columnas
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nVariables categóricas: {len(categorical_cols)}")
print(f"Variables numéricas: {len(numerical_cols)}")

# Codificar variables categóricas
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Limpiar valores extremos en numéricas
for col in numerical_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')
    upper_limit = X[col].quantile(0.99)
    X[col] = np.where(X[col] > upper_limit, upper_limit, X[col])
    X[col].fillna(X[col].median(), inplace=True)

# Guardar los nombres de LAS COLUMNAS EN EL ORDEN CORRECTO
# Esto es CRÍTICO para que la predicción funcione
columnas_ordenadas = categorical_cols + numerical_cols
print(f"\n✓ Columnas en el orden correcto: {len(columnas_ordenadas)} columnas")

# Asegurar que X tenga las columnas en ese orden
X = X[columnas_ordenadas]

# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTamaño entrenamiento: {X_train.shape[0]} registros")
print(f"Tamaño prueba: {X_test.shape[0]} registros")
print(f"Proporción hospitalizados en entrenamiento: {y_train.mean():.2%}")

DATASET CARGADO Y LIMPIO
Total de registros: 994
Hospitalizados (1): 377
No hospitalizados (0): 617

Variables categóricas: 11
Variables numéricas: 11

✓ Columnas en el orden correcto: 22 columnas

Tamaño entrenamiento: 795 registros
Tamaño prueba: 199 registros
Proporción hospitalizados en entrenamiento: 37.99%


# **Celda 2: Modelo Random Forest**

In [ ]:
# Celda 2: Modelo Random Forest
from sklearn.ensemble import RandomForestClassifier

print("="*60)
print("ENTRENANDO RANDOM FOREST")
print("="*60)

# Entrenar modelo
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

start_time = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start_time

# Predicciones
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métricas
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_proba_rf)

print("\n=== RANDOM FOREST - RESULTADOS ===")
print(f"✓ Accuracy:  {rf_accuracy:.4f}")
print(f"✓ Precision: {rf_precision:.4f}")
print(f"✓ Recall:    {rf_recall:.4f}")
print(f"✓ F1-Score:  {rf_f1:.4f}")
print(f"✓ AUC-ROC:   {rf_auc:.4f}")
print(f"⏱ Tiempo de entrenamiento: {rf_time:.4f} segundos")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_rf, target_names=['No Hospitalizado', 'Hospitalizado']))

# Guardar métricas
rf_metrics = {
    'Modelo': 'Random Forest',
    'Accuracy': rf_accuracy,
    'Precision': rf_precision,
    'Recall': rf_recall,
    'F1-Score': rf_f1,
    'AUC-ROC': rf_auc,
    'Tiempo (s)': rf_time
}

ENTRENANDO RANDOM FOREST

=== RANDOM FOREST - RESULTADOS ===
✓ Accuracy:  0.7487
✓ Precision: 0.6712
✓ Recall:    0.6533
✓ F1-Score:  0.6622
✓ AUC-ROC:   0.8035
⏱ Tiempo de entrenamiento: 0.2914 segundos

Matriz de confusión:
[[100  24]
 [ 26  49]]

Reporte de clasificación:
                  precision    recall  f1-score   support

No Hospitalizado       0.79      0.81      0.80       124
   Hospitalizado       0.67      0.65      0.66        75

        accuracy                           0.75       199
       macro avg       0.73      0.73      0.73       199
    weighted avg       0.75      0.75      0.75       199



# **Celda 3: Modelo Regresión Logística**

In [ ]:
# Celda 3: Modelo Regresión Logística
from sklearn.linear_model import LogisticRegression

print("="*60)
print("ENTRENANDO REGRESIÓN LOGÍSTICA")
print("="*60)

# Entrenar modelo
lr_model = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='liblinear'
)

start_time = time.time()
lr_model.fit(X_train, y_train)
lr_time = time.time() - start_time

# Predicciones
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Métricas
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_proba_lr)

print("\n=== REGRESIÓN LOGÍSTICA - RESULTADOS ===")
print(f"✓ Accuracy:  {lr_accuracy:.4f}")
print(f"✓ Precision: {lr_precision:.4f}")
print(f"✓ Recall:    {lr_recall:.4f}")
print(f"✓ F1-Score:  {lr_f1:.4f}")
print(f"✓ AUC-ROC:   {lr_auc:.4f}")
print(f"⏱ Tiempo de entrenamiento: {lr_time:.4f} segundos")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_lr, target_names=['No Hospitalizado', 'Hospitalizado']))

# Guardar métricas
lr_metrics = {
    'Modelo': 'Regresión Logística',
    'Accuracy': lr_accuracy,
    'Precision': lr_precision,
    'Recall': lr_recall,
    'F1-Score': lr_f1,
    'AUC-ROC': lr_auc,
    'Tiempo (s)': lr_time
}

ENTRENANDO REGRESIÓN LOGÍSTICA

=== REGRESIÓN LOGÍSTICA - RESULTADOS ===
✓ Accuracy:  0.7638
✓ Precision: 0.6628
✓ Recall:    0.7600
✓ F1-Score:  0.7081
✓ AUC-ROC:   0.8357
⏱ Tiempo de entrenamiento: 0.0069 segundos

Matriz de confusión:
[[95 29]
 [18 57]]

Reporte de clasificación:
                  precision    recall  f1-score   support

No Hospitalizado       0.84      0.77      0.80       124
   Hospitalizado       0.66      0.76      0.71        75

        accuracy                           0.76       199
       macro avg       0.75      0.76      0.75       199
    weighted avg       0.77      0.76      0.77       199



# **Celda 4: Validación con 10 pacientes nuevos (sin conocer su hospitalización)**

In [ ]:
# Celda 4: Validación con 10 pacientes DIFERENTES cada vez que se ejecuta
import random

print("="*60)
print("VALIDACIÓN CON 10 PACIENTES (ALEATORIOS CADA VEZ)")
print("="*60)

# Función para preprocesar un nuevo paciente
def preprocesar_nuevo_paciente(df_nuevo):
    """Preprocesa un nuevo paciente manteniendo el orden correcto de columnas"""
    df_procesado = pd.DataFrame(columns=columnas_ordenadas)

    for col in columnas_ordenadas:
        if col in df_nuevo.columns:
            df_procesado[col] = df_nuevo[col].values
        else:
            df_procesado[col] = 0

    for col in categorical_cols:
        if col in df_procesado.columns:
            le = LabelEncoder()
            df_procesado[col] = le.fit_transform(df_procesado[col].astype(str))

    for col in numerical_cols:
        df_procesado[col] = pd.to_numeric(df_procesado[col], errors='coerce').fillna(0)

    X_scaled = scaler.transform(df_procesado)
    return X_scaled

# SELECCIÓN ALEATORIA SIN SEMILLA FIJA (cambia cada vez)
# Eliminamos random_state para que sea diferente en cada ejecución
np.random.seed(None)  # Esto hace que sea realmente aleatorio cada vez
# O aún mejor, usamos random sin semilla:
pacientes_prueba = df.sample(n=10, random_state=None).copy()

# Guardar valores REALES
reales_hospitalizacion = pacientes_prueba['hospitalizacion'].values

# Eliminar la columna objetivo
pacientes_sin_etiqueta = pacientes_prueba.drop('hospitalizacion', axis=1)

print("\n📋 PACIENTES SELECCIONADOS (NUEVOS CADA VEZ):")
print("-"*60)
for i, (idx, row) in enumerate(pacientes_prueba.iterrows(), 1):
    print(f"Paciente {i}: Edad={row['edad']}, Género={row['genero']}, "
          f"IMC={row['imc']:.1f}, Diabetes={row['diabetes_previa']}, "
          f"Hipertensión={row['hipertension_previa']}")

# Predecir UNO POR UNO
predicciones_rf = []
probabilidades_rf = []
predicciones_lr = []
probabilidades_lr = []

for i in range(len(pacientes_sin_etiqueta)):
    un_paciente = pacientes_sin_etiqueta.iloc[[i]]
    X_nuevo = preprocesar_nuevo_paciente(un_paciente)

    pred_rf = rf_model.predict(X_nuevo)[0]
    prob_rf = rf_model.predict_proba(X_nuevo)[0, 1]

    pred_lr = lr_model.predict(X_nuevo)[0]
    prob_lr = lr_model.predict_proba(X_nuevo)[0, 1]

    predicciones_rf.append(pred_rf)
    probabilidades_rf.append(prob_rf)
    predicciones_lr.append(pred_lr)
    probabilidades_lr.append(prob_lr)

# Tabla comparativa
print("\n📊 PREDICCIONES VS REALIDAD:")
print("="*95)
print(f"{'Pac':<4} {'REALIDAD':<18} {'RF PREDICCIÓN':<18} {'RF PROB':<10} {'LR PREDICCIÓN':<18} {'LR PROB':<10} {'RF':<5} {'LR'}")
print("-"*95)

aciertos_rf = 0
aciertos_lr = 0

for i in range(10):
    realidad = "HOSPITALIZADO" if reales_hospitalizacion[i] == 1 else "NO HOSPITALIZADO"
    pred_rf = "HOSPITALIZADO" if predicciones_rf[i] == 1 else "NO HOSPITALIZADO"
    pred_lr = "HOSPITALIZADO" if predicciones_lr[i] == 1 else "NO HOSPITALIZADO"

    acierto_rf = "✓" if predicciones_rf[i] == reales_hospitalizacion[i] else "✗"
    acierto_lr = "✓" if predicciones_lr[i] == reales_hospitalizacion[i] else "✗"

    if predicciones_rf[i] == reales_hospitalizacion[i]:
        aciertos_rf += 1
    if predicciones_lr[i] == reales_hospitalizacion[i]:
        aciertos_lr += 1

    print(f"{i+1:<4} {realidad:<18} {pred_rf:<18} {probabilidades_rf[i]:<10.2%} {pred_lr:<18} {probabilidades_lr[i]:<10.2%} {acierto_rf:<5} {acierto_lr}")

print("="*95)

# Precisión
print("\n🎯 PRECISIÓN EN ESTOS 10 PACIENTES:")
print("-"*50)
print(f"✅ Random Forest:       {aciertos_rf}/10 aciertos = {aciertos_rf/10:.0%}")
print(f"✅ Regresión Logística: {aciertos_lr}/10 aciertos = {aciertos_lr/10:.0%}")

# Matrices de confusión
from sklearn.metrics import confusion_matrix

print("\n📈 MATRIZ DE CONFUSIÓN - RANDOM FOREST:")
cm_rf = confusion_matrix(reales_hospitalizacion, predicciones_rf)
print(cm_rf)
print(f"  VP={cm_rf[1,1]} | FN={cm_rf[1,0]} | VN={cm_rf[0,0]} | FP={cm_rf[0,1]}")

print("\n📈 MATRIZ DE CONFUSIÓN - REGRESIÓN LOGÍSTICA:")
cm_lr = confusion_matrix(reales_hospitalizacion, predicciones_lr)
print(cm_lr)
print(f"  VP={cm_lr[1,1]} | FN={cm_lr[1,0]} | VN={cm_lr[0,0]} | FP={cm_lr[0,1]}")

# Veredicto
print("\n🏆 RESULTADO EN ESTOS 10 PACIENTES:")
print("-"*40)
if aciertos_rf > aciertos_lr:
    print("✅ RANDOM FOREST fue más preciso en este grupo")
    print(f"   Ventaja: {aciertos_rf - aciertos_lr} acierto(s) más")
elif aciertos_lr > aciertos_rf:
    print("✅ REGRESIÓN LOGÍSTICA fue más precisa en este grupo")
    print(f"   Ventaja: {aciertos_lr - aciertos_rf} acierto(s) más")
else:
    print("⚖️ ¡EMPATE! Ambos modelos tuvieron el mismo desempeño")

print("\n💡 NOTA: Cada vez que ejecutes esta celda, se seleccionarán 10 pacientes DIFERENTES")

VALIDACIÓN CON 10 PACIENTES (ALEATORIOS CADA VEZ)

📋 PACIENTES SELECCIONADOS (NUEVOS CADA VEZ):
------------------------------------------------------------
Paciente 1: Edad=82, Género=F, IMC=22.2, Diabetes=NO, Hipertensión=SI
Paciente 2: Edad=20, Género=M, IMC=27.6, Diabetes=NO, Hipertensión=NO
Paciente 3: Edad=48, Género=F, IMC=30.5, Diabetes=SI, Hipertensión=SI
Paciente 4: Edad=49, Género=M, IMC=34.0, Diabetes=NO, Hipertensión=SI
Paciente 5: Edad=63, Género=M, IMC=36.2, Diabetes=SI, Hipertensión=NO
Paciente 6: Edad=84, Género=F, IMC=20.6, Diabetes=SI, Hipertensión=SI
Paciente 7: Edad=36, Género=M, IMC=25.8, Diabetes=NO, Hipertensión=SI
Paciente 8: Edad=73, Género=F, IMC=25.3, Diabetes=NO, Hipertensión=SI
Paciente 9: Edad=25, Género=M, IMC=25.8, Diabetes=SI, Hipertensión=SI
Paciente 10: Edad=22, Género=F, IMC=23.8, Diabetes=NO, Hipertensión=SI

📊 PREDICCIONES VS REALIDAD:
Pac  REALIDAD           RF PREDICCIÓN      RF PROB    LR PREDICCIÓN      LR PROB    RF    LR
--------------------

In [ ]:
# Celda 6: Tabla comparativa de métricas
import pandas as pd

# Recolectar métricas de ambos modelos
# (Estas variables ya deben existir de las celdas 2 y 3)

try:
    # Intentar obtener las métricas de Random Forest (Celda 2)
    rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))
    rf_precision = precision_score(y_test, rf_model.predict(X_test))
    rf_recall = recall_score(y_test, rf_model.predict(X_test))
    rf_f1 = f1_score(y_test, rf_model.predict(X_test))
except:
    # Si no existen, calcularlas
    rf_pred = rf_model.predict(X_test)
    rf_accuracy = accuracy_score(y_test, rf_pred)
    rf_precision = precision_score(y_test, rf_pred)
    rf_recall = recall_score(y_test, rf_pred)
    rf_f1 = f1_score(y_test, rf_pred)

try:
    # Intentar obtener las métricas de Regresión Logística (Celda 3)
    lr_accuracy = accuracy_score(y_test, lr_model.predict(X_test))
    lr_precision = precision_score(y_test, lr_model.predict(X_test))
    lr_recall = recall_score(y_test, lr_model.predict(X_test))
    lr_f1 = f1_score(y_test, lr_model.predict(X_test))
except:
    # Si no existen, calcularlas
    lr_pred = lr_model.predict(X_test)
    lr_accuracy = accuracy_score(y_test, lr_pred)
    lr_precision = precision_score(y_test, lr_pred)
    lr_recall = recall_score(y_test, lr_pred)
    lr_f1 = f1_score(y_test, lr_pred)

# Crear la tabla
tabla_comparativa = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest'],
    'Accuracy': [f"{lr_accuracy:.4f}", f"{rf_accuracy:.4f}"],
    'Precision': [f"{lr_precision:.4f}", f"{rf_precision:.4f}"],
    'Recall': [f"{lr_recall:.4f}", f"{rf_recall:.4f}"],
    'F1-score': [f"{lr_f1:.4f}", f"{rf_f1:.4f}"]
})

# Mostrar la tabla
print("\n" + "="*70)
print("📊 COMPARACIÓN DE MÉTRICAS - RANDOM FOREST vs REGRESIÓN LOGÍSTICA")
print("="*70)
print(tabla_comparativa.to_string(index=False))
print("="*70)

# También mostrar el resumen en porcentajes
print("\n📈 RESULTADOS EN PORCENTAJES:")
print("-"*50)
print(f"Regresión Logística: Accuracy={lr_accuracy:.1%} | Precision={lr_precision:.1%} | Recall={lr_recall:.1%} | F1={lr_f1:.1%}")
print(f"Random Forest:       Accuracy={rf_accuracy:.1%} | Precision={rf_precision:.1%} | Recall={rf_recall:.1%} | F1={rf_f1:.1%}")

# Indicar qué modelo es mejor en cada métrica
print("\n🏆 MODELO GANADOR POR MÉTRICA:")
print("-"*50)

if rf_accuracy > lr_accuracy:
    print(f"✅ Accuracy:  Random Forest ({rf_accuracy:.1%} vs {lr_accuracy:.1%})")
else:
    print(f"✅ Accuracy:  Regresión Logística ({lr_accuracy:.1%} vs {rf_accuracy:.1%})")

if rf_precision > lr_precision:
    print(f"✅ Precision: Random Forest ({rf_precision:.1%} vs {lr_precision:.1%})")
else:
    print(f"✅ Precision: Regresión Logística ({lr_precision:.1%} vs {rf_precision:.1%})")

if rf_recall > lr_recall:
    print(f"✅ Recall:    Random Forest ({rf_recall:.1%} vs {lr_recall:.1%})")
else:
    print(f"✅ Recall:    Regresión Logística ({lr_recall:.1%} vs {rf_recall:.1%})")

if rf_f1 > lr_f1:
    print(f"✅ F1-score:  Random Forest ({rf_f1:.1%} vs {lr_f1:.1%})")
else:
    print(f"✅ F1-score:  Regresión Logística ({lr_f1:.1%} vs {rf_f1:.1%})")


📊 COMPARACIÓN DE MÉTRICAS - RANDOM FOREST vs REGRESIÓN LOGÍSTICA
             Modelo Accuracy Precision Recall F1-score
Regresión Logística   0.7638    0.6628 0.7600   0.7081
      Random Forest   0.7487    0.6712 0.6533   0.6622

📈 RESULTADOS EN PORCENTAJES:
--------------------------------------------------
Regresión Logística: Accuracy=76.4% | Precision=66.3% | Recall=76.0% | F1=70.8%
Random Forest:       Accuracy=74.9% | Precision=67.1% | Recall=65.3% | F1=66.2%

🏆 MODELO GANADOR POR MÉTRICA:
--------------------------------------------------
✅ Accuracy:  Regresión Logística (76.4% vs 74.9%)
✅ Precision: Random Forest (67.1% vs 66.3%)
✅ Recall:    Regresión Logística (76.0% vs 65.3%)
✅ F1-score:  Regresión Logística (70.8% vs 66.2%)
